# 02 · 动作晚到之后，怎样定位失败并尝试恢复？

本课只改变命令到执行之间的时间，观察反馈何时失效，再在**相同延迟和初始状态**下尝试降低目标速度。
你要给出一个有对照、有适用范围的结论。先完成 [上一课](01_drive_and_observe.ipynb) 的手算与复核。

<figure class="course-figure">
  <img src="../../assets/visuals/feedback-loop.png" width="1536" height="1024" style="max-width:100%;height:auto" alt="原理图：控制器产生的动作进入四步延迟队列，在第 4 步执行更早发出的动作">
  <a href="../../assets/visuals/feedback-loop.png">查看原图</a>
  <figcaption><strong>AI 原理图 · 手算/机制示意</strong> · 延迟机制预览（第02课）</figcaption>
</figure>

**因果链**：当前状态 → 控制器命令 → 延迟队列 → `env.step` 中实际执行的动作 → 下一状态。每个决策间隔 `Δt=0.1s`，`d=4` 对应 `0.4s`；零开始编号的 `t=4` 是第 5 个决策间隔，执行 `u[0]`，前 4 个间隔执行零动作。

**手算检查**：若 `d=4`，第 4 步执行 `u[4]` 还是 `u[0]`？
<details><summary>展开答案</summary><p>执行 <code>u[0]</code>。一般第 <code>t</code> 步执行 <code>u[t-d]</code>；前 4 个决策间隔执行零动作。</p></details>

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.driving import DrivingConfig, run_episode, save_episode
OUTPUT = ROOT / "artifacts" / "first_loop"

## 1. 延迟不是随机噪声

第 t 步发出 u[t]，延迟 d 步后实际执行 u[t−d]；控制器仍用当前状态算新命令。
队列未填满时执行 `[0,0]`。d=4、dt=0.1秒，对应 **0.4秒**：

| 执行区间开始 | 发出 | 执行 |
|---|---|---|
| 0.0s | u[0] | [0,0] |
| 0.1s | u[1] | [0,0] |
| 0.3s | u[3] | [0,0] |
| 0.4s | u[4] | u[0] |

原本纠正右偏的左转命令，晚到时车辆可能已经接近中心，继续左转就可能过冲。
本实验同时延迟转向和油门，初始等待也改变速度过程。若要单独研究转向延迟，应另外设计执行器和对照。

## 2. 三组条件与预测

| 组别 | 道路、偏移、种子、时长 | 延迟 | 目标速度 |
|---|---|---|---|
| baseline | 相同 | 0 步 | 6 m/s |
| delay | 相同 | 4 步 | 6 m/s |
| recovery | 相同 | 4 步 | 2 m/s |

baseline→delay 只改变延迟，delay→recovery 只改变目标速度。“恢复方案”是重新运行同一失败条件时的减速策略，
并非检测到碰撞后倒退回安全状态，也不保证能应对任意延迟。

先写下：延迟可能怎样改变第一次转向、回正时间和失败标志？减速若减少过冲，会付出什么代价？

In [ ]:
base = DrivingConfig()
configs = {"baseline": base, "delay": replace(base, action_delay_steps=4),
           "recovery": replace(base, action_delay_steps=4, target_speed_mps=2.0)}
results, saved = {}, {}
for name, cfg in configs.items():
    results[name] = run_episode(cfg)
    saved[name] = save_episode(results[name], OUTPUT, name, failure_gif=True)
columns = ["steps", "elapsed_s", "distance_traveled_m", "mean_abs_lateral_error_m",
           "max_abs_lateral_error_m", "failure_reason", "outcome", "route_completion"]
display(pd.DataFrame({n: r.metrics for n, r in results.items()}).T[columns])

## 3. 先核对动作链，再解释成绩

若 delay 第4行实际动作不等于第0行命令，先检查实现，再讨论延迟效果。
横向误差测量执行后的车辆中心；出界按模拟器车身/路面接触等规则判定，两者测量对象不同。

In [ ]:
delayed = results["delay"]
d = configs["delay"].action_delay_steps
commands = np.array([[r["command_steering"], r["command_throttle"]] for r in delayed.trace])
applied = np.array([[r["applied_steering"], r["applied_throttle"]] for r in delayed.trace])
assert np.allclose(applied[:d], 0)
assert np.allclose(applied[d:], commands[:-d])
display(pd.DataFrame(delayed.trace).head(9)[[
    "step", "time_s", "before_lateral_error_m", "command_steering", "applied_steering", "lateral_error_m"]])
fig, axes = plt.subplots(2, 1, figsize=(10, 6), constrained_layout=True)
for name, result in results.items():
    frame = pd.DataFrame(result.trace)
    axes[0].plot(frame.time_s, frame.lateral_error_m, label=name)
    axes[1].plot(frame.time_s, frame.speed_mps, label=name)
axes[0].set(ylabel="Lateral error / m")
axes[1].set(xlabel="Time / s", ylabel="Speed / m/s")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.show()

## 4. 失败现场：车的中心不是整辆车

矩形车身长度 a、宽度 b、朝向 theta 时，横向包络半宽约为
`a/2 * abs(sin(theta)) + b/2 * abs(cos(theta))`。
所以车中心离中心线不远，斜着行驶的前角也可能先碰连续线。这是几何近似；确切终止原因以本次接触标志为证据。

下方显示失败最后几行，播放基于真实轨迹和车身尺寸的俯视回放。GIF 是轨迹可视化，不是相机视频。
参数改变后若没有失败，就如实报告该次未出现失败。

In [ ]:
for name, result in results.items():
    if result.metrics["failure"]:
        print(name, result.metrics["failure_reason"])
        display(pd.DataFrame(result.trace).tail(4)[[
            "time_s", "lateral_error_m", "heading_rad", "on_lane",
            "on_white_continuous_line", "on_yellow_continuous_line", "crash_sidewalk"]])
        display(Image(filename=str(saved[name]["failure_gif"])))

## 5. 一个公平但有限的比较

baseline 走了更长时间，delay 可能提前失败，recovery 又可能因速度低走得短。先报告终止原因与距离，
再比较共同时间窗口中的误差。共同窗口避免平均值直接受不同时长影响，但仍不是同路程/同完成度比较。

outcome=arrived 表示到达，failure 表示触发失败，horizon 表示跑满时长。本课18秒内未到终点很正常。
跑满时长且未失败，只支持“这段试验内未触发失败”。
route_completion 是模拟器整条路线的累计位置比例；本课从直段中部起步，因此其中包含起点之前的路线，
不能把它直接当成本次行驶完成的比例。本次运动量看 distance_traveled_m。

In [ ]:
common_steps = min(len(r.trace) for r in results.values())
common = {}
for name, result in results.items():
    rows = result.trace[:common_steps]
    common[name] = {"common_time_s": rows[-1]["time_s"],
        "mean_abs_error_m": np.mean([abs(r["lateral_error_m"]) for r in rows]),
        "distance_m": sum(r["distance_m"] for r in rows)}
display(pd.DataFrame(common).T)

## 6. 独立实验与研究笔记

固定其余条件，只改延迟为1、2、8、12步；使用独立文件名保存，不只保留好看的曲线。再选未调参的初始偏移（如−0.4m），
检查减速方案能否迁移。道路固定且无交通，更换 seed 不等于泛化到新道路；偏移变化才是这里的具体分布变化。

一页笔记按此顺序交付：问题 → 运行前预测 → 唯一改动 → 配置/轨迹位置 → 完整结果 → 失败片段 →
机制解释 → 结论范围 → 下一项对照。预测不符时，写出仍待排查的因素。

<details><summary>推理与答案</summary>

延迟使动作基于过时状态；减速通常缩短相同延迟期间的位移，可能减少过冲，但不是稳定性证明。
减速降低进度，短时存活不足以证明任务完成。本次固定条件下 delay=12、target_speed=2 仍会失败；改变起点后要重新验证。

不能用 recovery 没出界单独证明有效，必须保留相同偏移、延迟、道路和时长的原策略对照。
不能给 recovery 去掉延迟或换容易的起点。平均误差小却提前失败，先看时间与接触标志。
最后一帧中心误差不大时，检查车身朝向、尺寸和车道边界；保留所有失败标志。
</details>

## 7. 与研究脉络的连接

后续可用几何控制器的状态—动作示范做行为克隆；但离线动作误差小，不代表策略在自己造成的新状态上能恢复。
DAgger 讨论这种状态分布变化。RL 再通过交互和奖励优化策略，仍面对执行、延迟和评测问题。

选读 [DAgger](https://arxiv.org/abs/1011.0686) 的 Introduction，限45分钟：用偏离—反馈例子解释
“专家访问的状态”和“学习者访问的状态”为何不同。论文结论与本课规则控制实验分别是什么证据？类比不等于论文复现。